# 클래스 확인문제 (심화)

10개의 심화 클래스 문제를 통해 파이썬 객체지향의 핵심 개념을 마스터하세요.

---
## 문제 1: MRO 다이아몬드 상속

다음 다이아몬드 상속 구조에서 `D().greet()`의 출력 결과를 예측하고, MRO(Method Resolution Order)가 왜 그 순서인지 설명하세요.

```python
class A:
    def greet(self):
        return "A"

class B(A):
    def greet(self):
        return "B"

class C(A):
    def greet(self):
        return "C"

class D(B, C):
    pass

print(D().greet())
print(D.__mro__)
```

**요구사항:**
1. 출력 결과를 예측하시오.
2. `D(B, C)`를 `D(C, B)`로 바꾸면 결과가 어떻게 달라지는지 설명하시오.
3. `super().greet()`를 활용하여 `D.greet()`에서 B와 C의 greet를 모두 호출하는 코드를 작성하시오.

In [ ]:
# 문제 1 - 풀이

class A:
    def greet(self):
        return "A"

class B(A):
    def greet(self):
        return "B"

class C(A):
    def greet(self):
        return "C"

class D(B, C):
    pass

# 1. 출력 예측
print("D().greet():", D().greet())
print("MRO:", [cls.__name__ for cls in D.__mro__])

# 2. D(C, B)로 변경 시
class D2(C, B):
    pass

print("D2().greet():", D2().greet())
print("MRO:", [cls.__name__ for cls in D2.__mro__])

# 3. super()로 B와 C 모두 호출 (cooperative multiple inheritance)
class A_coop:
    def greet(self):
        print("A")

class B_coop(A_coop):
    def greet(self):
        print("B")
        super().greet()

class C_coop(A_coop):
    def greet(self):
        print("C")
        super().greet()

class D_coop(B_coop, C_coop):
    def greet(self):
        print("D")
        super().greet()

D_coop().greet()
print("D_coop MRO:", [cls.__name__ for cls in D_coop.__mro__])

---
## 문제 2: Property 검증

`Temperature` 클래스를 구현하세요. `celsius` 프로퍼티는 -273.15(절대영도) 미만의 값을 설정하려고 하면 `ValueError`를 발생시켜야 합니다. `fahrenheit` 프로퍼티도 제공하여 화씨로도 읽고 쓸 수 있게 하세요.

**조건:**
- `celsius` setter에서 절대영도 미만 값 거부
- `fahrenheit` 프로퍼티는 `celsius`를 기반으로 자동 변환
- `fahrenheit = celsius * 9/5 + 32`

In [ ]:
# 문제 2 - 풀이

class Temperature:
    ABSOLUTE_ZERO_C = -273.15

    def __init__(self, celsius=0):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < self.ABSOLUTE_ZERO_C:
            raise ValueError(f"온도가 절대영도({self.ABSOLUTE_ZERO_C}°C) 미만입니다: {value}")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = (value - 32) * 5 / 9

# 테스트
t = Temperature(25)
print(f"25°C = {t.fahrenheit}°F")

t.fahrenheit = 212
print(f"212°F = {t.celsius}°C")

try:
    t.celsius = -300
except ValueError as e:
    print(f"ValueError: {e}")

try:
    t.fahrenheit = -500
except ValueError as e:
    print(f"ValueError: {e}")

---
## 문제 3: Dataclass 상속

`dataclass`를 사용하여 다음 요구사항을 만족하는 클래스 계층을 구현하세요.

**조건:**
1. `Employee` 기본 데이터클래스: `name: str`, `id: int`, `salary: float`
2. `Manager(Employee)` 상속 데이터클래스: `department: str`, `direct_reports: list` (기본값 빈 리스트)
3. `direct_reports`는 `Employee` 객체의 리스트
4. `Manager`에 `total_salary` 메서드: 본인 급여 + 직속 부하 급여 합계 반환

**주의:** dataclass 상속 시 필드 순서 규칙과 `list` 기본값 문제를 해결해야 합니다.

In [ ]:
# 문제 3 - 풀이

from dataclasses import dataclass, field

@dataclass
class Employee:
    name: str
    id: int
    salary: float

@dataclass
class Manager(Employee):
    department: str = ""
    direct_reports: list = field(default_factory=list)

    def total_salary(self) -> float:
        return self.salary + sum(e.salary for e in self.direct_reports)

# 테스트
e1 = Employee("김철수", 101, 50000)
e2 = Employee("이영희", 102, 60000)
e3 = Employee("박지성", 103, 55000)

m = Manager("최관리", 1, 100000, "개발팀", [e1, e2, e3])

print(m)
print(f"팀 전체 급여: ${m.total_salary():,.0f}")
print(f"  본인: ${m.salary:,.0f}")
print(f"  직원 합계: ${sum(e.salary for e in m.direct_reports):,.0f}")

---
## 문제 4: `__eq__` / `__lt__` 정렬

`Student` 클래스를 구현하여 학생 리스트를 (1) 이름순, (2) 점수 내림차순으로 정렬할 수 있게 하세요.

**조건:**
- `__eq__`와 `__lt__`를 구현하여 점수 기준 비교 가능 (높은 점수가 "더 작게" 비교되도록 → 내림차순)
- `__repr__` 구현
- `functools.total_ordering` 데코레이터 사용
- 이름순 정렬은 `key` 인자로 처리

In [ ]:
# 문제 4 - 풀이

from functools import total_ordering

@total_ordering
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    def __eq__(self, other):
        if not isinstance(other, Student):
            return NotImplemented
        return self.score == other.score

    def __lt__(self, other):
        if not isinstance(other, Student):
            return NotImplemented
        return self.score > other.score  # 높은 점수가 "더 작게" → 내림차순

    def __repr__(self):
        return f"Student('{self.name}', {self.score})"

# 테스트
students = [
    Student("박지성", 88),
    Student("김철수", 95),
    Student("이영희", 88),
    Student("최민수", 72),
    Student("정다은", 95),
]

print("=== 점수 내림차순 ===")
for s in sorted(students):
    print(f"  {s}")

print("\n=== 이름순 ===")
for s in sorted(students, key=lambda s: s.name):
    print(f"  {s}")

# 비교 테스트
print(f"\n김철수(95) == 정다은(95): {students[1] == students[4]}")
print(f"김철수(95) < 박지성(88): {students[1] < students[0]}")  # 95가 더 높으므로 True

---
## 문제 5: ABC 강제 구현

`abc` 모듈을 사용하여 `Shape` 추상 기반 클래스를 만들고, 이를 상속받는 `Circle`과 `Rectangle`을 구현하세요.

**조건:**
- `Shape`은 `area()`와 `perimeter()` 추상 메서드를 가져야 함
- `Shape`에 `__repr__` 구현: `"{클래스명}(면적={area값}, 둘레={perimeter값})"`
- `Circle(radius)`와 `Rectangle(width, height)` 구현
- `Shape`을 직접 인스턴스화하려고 하면 `TypeError` 발생 확인
- 추상 메서드를 누락한 서브클래스 인스턴스화 시 `TypeError` 확인

In [ ]:
# 문제 5 - 풀이

from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self) -> float:
        ...

    @abstractmethod
    def perimeter(self) -> float:
        ...

    def __repr__(self):
        return f"{self.__class__.__name__}(면적={self.area():.2f}, 둘레={self.perimeter():.2f})"

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)

# 테스트
c = Circle(5)
r = Rectangle(4, 6)
print(c)
print(r)

# Shape 직접 인스턴스화 → TypeError
try:
    Shape()
except TypeError as e:
    print(f"\nShape() TypeError: {e}")

# 추상 메서드 누락 서브클래스 → TypeError
class IncompleteShape(Shape):
    def area(self):
        return 0
    # perimeter 누락!

try:
    IncompleteShape()
except TypeError as e:
    print(f"IncompleteShape() TypeError: {e}")

---
## 문제 6: 컨텍스트 매니저 클래스

파일 읽기/쓰기 시간을 측정하는 `Timer` 컨텍스트 매니저 클래스를 `__enter__`/`__exit__`로 구현하세요.

**조건:**
- `with Timer("작업명") as t:` 형태로 사용
- 진입 시 시작 시간 기록, 종료 시 경과 시간 출력
- `__exit__`에서 예외가 발생해도 경과 시간은 출력
- 예외가 발생한 경우 어떤 예외였는지 함께 출력
- `t.elapsed` 속성으로 경과 시간(초) 접근 가능

In [ ]:
# 문제 6 - 풀이

import time

class Timer:
    def __init__(self, label="작업"):
        self.label = label
        self.start = None
        self.elapsed = 0

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"[{self.label}] 경과 시간: {self.elapsed:.4f}초")
        if exc_type is not None:
            print(f"[{self.label}] 예외 발생: {exc_type.__name__}: {exc_val}")
        return False  # 예외를 억제하지 않음

# 테스트 1: 정상 종료
with Timer("합계 계산") as t:
    total = sum(range(10_000_000))
print(f"elapsed 속성: {t.elapsed:.4f}초\n")

# 테스트 2: 예외 발생
try:
    with Timer("예외 테스트") as t:
        raise ValueError("의도한 오류")
except ValueError:
    pass

# 테스트 3: 중첩 사용
with Timer("외부") as t1:
    time.sleep(0.05)
    with Timer("내부") as t2:
        time.sleep(0.03)
    print(f"  내부 경과: {t2.elapsed:.4f}초")
print(f"외부 경과: {t1.elapsed:.4f}초")

---
## 문제 7: 이터레이터 프로토콜

`__iter__`와 `__next__`를 구현하여 피보나치 수열 이터레이터 클래스 `FibIterator`를 만드세요.

**조건:**
- `FibIterator(limit)`: `limit`개의 피보나치 수를 생성
- `__iter__`은 `self` 반환
- `__next__`에서 `limit`에 도달하면 `StopIteration` 발생
- `iter()`와 `next()` 내장 함수로 동작 확인
- `for` 루프에서도 동작 확인
- 이터레이터는 소진 후 재사용할 수 없음을 보여라

In [ ]:
# 문제 7 - 풀이

class FibIterator:
    def __init__(self, limit):
        self.limit = limit
        self.count = 0
        self.prev = 0
        self.curr = 1

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        if self.count == 0:
            self.count += 1
            return 0
        result = self.curr
        self.prev, self.curr = self.curr, self.prev + self.curr
        self.count += 1
        return result

# 테스트 1: iter/next 사용
fib = FibIterator(10)
it = iter(fib)
print("iter/next:", [next(it) for _ in range(10)])

# 테스트 2: for 루프
fib2 = FibIterator(10)
print("for 루프:", list(fib2))

# 테스트 3: 소진 후 재사용 불가
fib3 = FibIterator(5)
print("첫 순회:", list(fib3))
print("재사용:", list(fib3))  # 빈 리스트 → 이미 소진됨

---
## 문제 8: 클래스메서드 대체 생성자

`Date` 클래스를 구현하세요. 다양한 형식의 입력으로 인스턴스를 생성할 수 있는 대체 생성자(classmethod)를 제공합니다.

**조건:**
- 기본 생성자: `Date(year, month, day)`
- `Date.from_string("2025-01-15")`: ISO 형식 문자열에서 생성
- `Date.from_timestamp(1705276800)`: Unix 타임스탬프에서 생성
- `Date.today()`: 오늘 날짜
- `__repr__` 구현
- 잘못된 형식의 문자열은 `ValueError` 발생

In [ ]:
# 문제 8 - 풀이

import datetime

class Date:
    def __init__(self, year, month, day):
        self.year = year
        self.month = month
        self.day = day

    def __repr__(self):
        return f"Date({self.year}, {self.month:02d}, {self.day:02d})"

    @classmethod
    def from_string(cls, date_str):
        try:
            parts = date_str.split("-")
            if len(parts) != 3:
                raise ValueError
            year, month, day = int(parts[0]), int(parts[1]), int(parts[2])
            # 유효성 검증: datetime으로 검사
            datetime.date(year, month, day)
            return cls(year, month, day)
        except (ValueError, AttributeError) as e:
            raise ValueError(f"잘못된 날짜 형식: '{date_str}'. 'YYYY-MM-DD' 형식이어야 합니다.") from e

    @classmethod
    def from_timestamp(cls, ts):
        dt = datetime.datetime.fromtimestamp(ts)
        return cls(dt.year, dt.month, dt.day)

    @classmethod
    def today(cls):
        today = datetime.date.today()
        return cls(today.year, today.month, today.day)

# 테스트
d1 = Date(2025, 1, 15)
print(f"기본 생성자: {d1}")

d2 = Date.from_string("2025-06-20")
print(f"from_string: {d2}")

d3 = Date.from_timestamp(1705276800)
print(f"from_timestamp: {d3}")

d4 = Date.today()
print(f"today: {d4}")

# 잘못된 형식 테스트
try:
    Date.from_string("2025/06/20")
except ValueError as e:
    print(f"\nValueError: {e}")

try:
    Date.from_string("2025-13-01")
except ValueError as e:
    print(f"ValueError: {e}")

---
## 문제 9: `__repr__` vs `__str__`

`Product` 클래스에서 `__repr__`과 `__str__`의 차이를 실험하세요.

**조건:**
- `Product(name, price, stock)` 클래스
- `__repr__`: `eval()`로 복원 가능한 형태 → `Product('노트북', 1500000, 10)`
- `__str__`: 사용자 친화적 형태 → `노트북: ₩1,500,000 (재고: 10개)`
- 다음을 확인:
  1. `print(obj)` → `__str__` 호출
  2. `repr(obj)` → `__repr__` 호출
  3. 컨테이너(`list`) 안의 객체 → `__repr__` 호출
  4. `eval(repr(obj))`로 복원 가능
  5. `__str__`이 없을 때 `__repr__`이 대체하는지 확인

In [ ]:
# 문제 9 - 풀이

class Product:
    def __init__(self, name, price, stock):
        self.name = name
        self.price = price
        self.stock = stock

    def __repr__(self):
        return f"Product('{self.name}', {self.price}, {self.stock})"

    def __str__(self):
        return f"{self.name}: ₩{self.price:,} (재고: {self.stock}개)"

p = Product("노트북", 1500000, 10)

# 1. print → __str__
print("1. print():", p)

# 2. repr() → __repr__
print("2. repr():", repr(p))

# 3. 리스트 안의 객체 → __repr__
items = [Product("노트북", 1500000, 10), Product("마우스", 30000, 50)]
print("3. 리스트 안:", items)

# 4. eval(repr()) 복원
p_copy = eval(repr(p))
print("4. eval(repr(p)):", p_copy)
print("   복원 확인:", p_copy.name == p.name and p_copy.price == p.price)

# 5. __str__이 없을 때 __repr__ 대체
class ProductNoStr:
    def __init__(self, name, price, stock):
        self.name = name
        self.price = price
        self.stock = stock

    def __repr__(self):
        return f"ProductNoStr('{self.name}', {self.price}, {self.stock})"

p2 = ProductNoStr("모니터", 500000, 5)
print(f"5. __str__ 없이 print(): {p2}")  # __repr__이 대체

---
## 문제 10: 컴포지션 패턴

상속 대신 컴포지션을 사용하여 `Car` 클래스를 설계하세요. `Car`은 `Engine`과 `GPS`를 **가지고(has-a)** 있으며, 상속이 아닙니다.

**조건:**
- `Engine` 클래스: `horsepower`, `fuel_type`, `start()`, `stop()` 메서드
- `GPS` 클래스: `latitude`, `longitude`, `navigate(dest)` 메서드
- `Car` 클래스: `Engine`과 `GPS` 객체를 컴포지션으로 포함
  - `Car(make, model, engine, gps)` 생성
  - `start_engine()`: 엔진 시작 위임
  - `navigate_to(dest)`: GPS 내비게이션 위임
  - `__repr__` 구현
- 상속 없이 구현할 것
- 엔진이 정지 상태에서 `navigate_to` 호출 시 경고 메시지 출력

In [ ]:
# 문제 10 - 풀이

class Engine:
    def __init__(self, horsepower, fuel_type):
        self.horsepower = horsepower
        self.fuel_type = fuel_type
        self.running = False

    def start(self):
        self.running = True
        return f"{self.horsepower}마력 {self.fuel_type} 엔진 시동 걸림 🏎️"

    def stop(self):
        self.running = False
        return f"엔진 정지 🛑"

    def __repr__(self):
        state = "시동 ON" if self.running else "시동 OFF"
        return f"Engine({self.horsepower}hp, {self.fuel_type}, {state})"

class GPS:
    def __init__(self, latitude=0.0, longitude=0.0):
        self.latitude = latitude
        self.longitude = longitude

    def navigate(self, dest):
        return f"({self.latitude:.2f}, {self.longitude:.2f}) → {dest}로 내비게이션 시작 🗺️"

    def __repr__(self):
        return f"GPS({self.latitude:.2f}, {self.longitude:.2f})"

class Car:
    def __init__(self, make, model, engine, gps):
        self.make = make
        self.model = model
        self.engine = engine  # 컴포지션
        self.gps = gps        # 컴포지션

    def start_engine(self):
        return self.engine.start()

    def stop_engine(self):
        return self.engine.stop()

    def navigate_to(self, dest):
        if not self.engine.running:
            print(f"⚠️ 경고: 엔진이 정지 상태입니다. 먼저 시동을 켜세요.")
        return self.gps.navigate(dest)

    def __repr__(self):
        return f"Car({self.make} {self.model}, {self.engine}, {self.gps})"

# 테스트
engine = Engine(300, "가솔린")
gps = GPS(37.5665, 126.9780)
car = Car("현대", "아반떼 N", engine, gps)

print(car)
print(car.start_engine())
print(car.navigate_to("부산 해운대"))
print(car.stop_engine())

print("\n--- 엔진 정지 상태에서 내비게이션 ---")
print(car.navigate_to("제주도"))

---
## 요약

| # | 주제 | 핵심 개념 |
|---|------|----------|
| 1 | MRO 다이아몬드 | C3 선형화, `super()` 협력적 다중상속 |
| 2 | Property 검증 | `@property`, `@x.setter`, 값 검증 |
| 3 | Dataclass 상속 | `field(default_factory=)`, 필드 순서 규칙 |
| 4 | `__eq__`/`__lt__` | `total_ordering`, 비교 프로토콜 |
| 5 | ABC 강제 구현 | `ABC`, `@abstractmethod`, 인터페이스 강제 |
| 6 | 컨텍스트 매니저 | `__enter__`/`__exit__`, 예외 처리 |
| 7 | 이터레이터 프로토콜 | `__iter__`/`__next__`, `StopIteration` |
| 8 | 클래스메서드 생성자 | `@classmethod`, 대체 생성자 패턴 |
| 9 | `__repr__` vs `__str__` | 디버그 vs 사용자 표현, `eval(repr())` |
| 10 | 컴포지션 패턴 | has-a 관계, 위임, 상속 대안 |